# 10.2 Setting up and solving models and data

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs', 'cbc'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To apply a solver to an instance of a model, the examples in this book use the `read()` method to read the model, Python data structures to supply the data, and the `solve()` method. For the diet model and data of [Chapter 2](../02/02.md):

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost:  sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
   n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing diet.mod


In [3]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 0, 100),
        ("CHK",  2.59, 0, 100),
        ("FISH", 2.29, 0, 100),
        ("HAM",  2.89, 0, 100),
        ("MCH",  1.89, 0, 100),
        ("MTL",  1.99, 0, 100),
        ("SPG",  1.99, 0, 100),
        ("TUR",  2.49, 0, 100),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

nutr_df = pd.DataFrame(
    [
        ("A",  700, 10000),
        ("C",  700, 10000),
        ("B1", 700, 10000),
        ("B2", 700, 10000),
    ],
    columns=["NUTR", "n_min", "n_max"],
).set_index("NUTR")

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15],
        "CHK":  [ 8,  0, 20, 20],
        "FISH": [ 8, 10, 15, 10],
        "HAM":  [40, 40, 35, 10],
        "MCH":  [15, 35, 15, 15],
        "MTL":  [70, 30, 15, 15],
        "SPG":  [25, 50, 25, 15],
        "TUR":  [60, 20, 15, 10],
    },
    index=["A", "C", "B1", "B2"],
)

In [4]:
ampl.read("diet.mod")

ampl.set_data(food_df, "FOOD")
ampl.set_data(nutr_df, "NUTR")
ampl.param["amt"] = amt_df.stack().to_dict()

ampl.solve(solver="highs")

HiGHS 1.15.1: optimal solution; objective 88.2
2 simplex iterations
0 barrier iterations


The `read()` method names a file that contains model declarations (Chapters 5 through 8), and the data values for model components are supplied from Python ([Chapter 9](../09/09.md)). The `solve()` method causes a description of the optimization problem to be sent to a solver, and the results to be retrieved for examination. This section takes a closer look at the main AMPL features for setting up and solving models. Features for subsequently changing and re-solving models are covered in [Section 10.3](10_3_modifying_data.ipynb) and [Section 10.4](10_4_modifying_models.ipynb).

## 10.2.1 Entering models and data

AMPL maintains a "current" model, which is the one that will be sent to the solver if you call `solve()`. At the beginning of a session, the current model is empty. The `read()` method reads declarations from a file and adds them to the current model; data assignments such as `set_data()` or `ampl.param[...]` supply values for components already in the current model. Thus you may use several `read()` calls and data assignments to build up the description of an optimization problem, reading different parts of the model from different files, and taking the data from different Python sources.

Model declarations such as `param`, `var` and `subject to` act as commands that add components to the current model, so a model can be split among several files, each read with its own call to `read()`. (Declarations could also be passed as strings to the `eval()` method, but keeping them in `.mod` files makes models easier to review and reuse.) If data is kept in files of AMPL data statements, the `read_data()` method reads them: it switches AMPL to "data mode" for the contents of the file, and then returns it to the "model mode" in which all other commands are interpreted.

If a model declares more than one objective function, AMPL by default passes all of them to the solver. Most solvers deal only with one objective function and usually select the first by default. The AMPL `objective` command lets you select a single objective function to pass to the solver; it consists of the keyword `objective` followed by a name from a `minimize` or `maximize` declaration. amplpy has no method of its own for this command, so it is given through `eval()`. As an example, we add to the diet model a second objective, read from its own file, that counts the total number of packages bought, and select it:

In [5]:
%%writefile number_obj.mod

minimize Total_Number: sum {j in FOOD} Buy[j];

Writing number_obj.mod


In [6]:
ampl.read("number_obj.mod")
ampl.eval("objective Total_Number;")

ampl.get_current_objective().name()

'Total_Number'

In [7]:
ampl.solve(solver="highs")
ampl.display("Total_Number", "Total_Cost")

HiGHS 1.15.1: optimal solution; objective 38.79277567
3 simplex iterations
0 barrier iterations
Total_Number = 38.7928
Total_Cost = 91.3706



Both objectives are evaluated at the new solution, but only `Total_Number` was minimized. If a model has an indexed collection of objectives, you must supply a subscript to indicate which one is to be chosen, as in `objective Total_Cost["A&P"];`. The uses of multiple objectives are illustrated by two examples in [Section 8.3](../08/8_3_objectives.ipynb). Before continuing, we return to the original objective:

In [8]:
ampl.eval("objective Total_Cost;")

## 10.2.2 Solving a model

The `solve()` method sets in motion a series of activities. First, it causes AMPL to generate a specific optimization problem from the model and data that you have supplied. If you have neglected to provide some needed data, an error message is printed; you will also get error messages if your data values violate any restrictions imposed by qualification phrases in `var` or `param` declarations or by `check` statements. AMPL waits to verify data restrictions until you call `solve()`, because a restriction may depend in a complicated way on many different data values. Arithmetic errors like dividing by zero are also caught at this stage.

After the optimization problem is generated, AMPL enters a "presolve" phase that tries to make the problem easier for the solver. Sometimes presolve so greatly reduces the size of a problem that it becomes substantially easier to solve. Normally the work of presolve goes on behind the scenes, however, and you need not be concerned about it. In rare cases, presolve can substantially affect the optimal values of the variables (when there is more than one optimal solution) or can interfere with other preprocessing routines that are built into your solver software. Also presolve sometimes detects that no feasible solution is possible, and so does not bother sending your problem to the solver. For example, if you drastically reduce the availability of one resource in `steel4.mod` of [Chapter 1](../01/01.md), then AMPL produces an error message:

In [9]:
%%writefile steel4.mod

set PROD;   # products
set STAGE;  # stages

param rate {PROD,STAGE} > 0; # tons per hour in each stage
param avail {STAGE} >= 0;    # hours available/week in each stage
param profit {PROD};         # profit per ton

param commit {PROD} >= 0;    # lower limit on tons sold in week
param market {PROD} >= 0;    # upper limit on tons sold in week

var Make {p in PROD} >= commit[p], <= market[p]; # tons produced

# Objective: total profits from all products
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];

# In each stage: total of hours used by all
# products may not exceed hours available
subject to Time {s in STAGE}:
   sum {p in PROD} (1/rate[p,s]) * Make[p] <= avail[s];

Writing steel4.mod


In [10]:
steel4 = AMPL()   # a second AMPL session, for the steel4 model
steel4.read("steel4.mod")

steel4.set_data(
    pd.DataFrame(
        [("bands", 25, 1000, 6000), ("coils", 30, 500, 4000), ("plate", 29, 750, 3500)],
        columns=["PROD", "profit", "commit", "market"],
    ).set_index("PROD"),
    "PROD",
)
steel4.set_data(
    pd.DataFrame([("reheat", 35), ("roll", 40)], columns=["STAGE", "avail"]).set_index("STAGE"),
    "STAGE",
)
steel4.param["rate"] = {
    ("bands", "reheat"): 200, ("bands", "roll"): 200,
    ("coils", "reheat"): 200, ("coils", "roll"): 140,
    ("plate", "reheat"): 200, ("plate", "roll"): 160,
}

steel4.param["avail"]["reheat"] = 10
steel4.solve(solver="highs")

[WARNING] 
	presolve: constraint Time['reheat'] cannot hold:
		body <= 10 cannot be >= 11.25; difference = -1.25


For these cases you should consult the detailed description of presolve in [Section 14.1](../14/14_1_presolve.ipynb).

## 10.2.3 Checking the result of a solve

At the interactive prompt, you read the solver's message and decide what to do next. A script must make that decision by itself, so it has to check the outcome of every solve. After each call to `solve()`, amplpy's `solve_result` and `solve_result_num` properties describe the solver's status at the end of its run, `solve_result_num` by a number and `solve_result` by a character string:

In [11]:
steel4.solve_result, steel4.solve_result_num

('infeasible', 299)

The possible values may be interpreted as follows (see [Section 14.2](../14/14_2_retrieving_results_from_solvers.ipynb) for details):

| `solve_result_num` | `solve_result` | interpretation |
| :--- | :--- | :--- |
| 0–99 | `solved` | optimal solution found |
| 100–199 | `solved?` | optimal solution indicated, but error likely |
| 200–299 | `infeasible` | constraints cannot be satisfied |
| 300–399 | `unbounded` | objective can be improved without limit |
| 400–499 | `limit` | stopped by a limit that you set (such as on iterations) |
| 500–599 | `failure` | stopped by an error condition in the solver |

Before the first solve, and after a solve that could not generate the problem (because some needed data is missing, for instance), `solve_result_num` is -1 and `solve_result` is `'?'`. Errors of this kind are printed, but they do not stop a Python program. The simplest safeguard is to assert the expected result after every solve, so that a script stops at the first unexpected outcome instead of going on to use meaningless values. With the original availability restored, the `steel4` problem solves again:

In [12]:
steel4.param["avail"]["reheat"] = 35
steel4.solve(solver="highs")
assert steel4.solve_result == "solved", steel4.solve_result

steel4.close()   # this session is no longer needed

HiGHS 1.15.1: optimal solution; objective 190071.4286
2 simplex iterations
0 barrier iterations


We follow this practice in the remainder of the book.

## 10.2.4 Choosing a solver

The generated optimization problem, as possibly modified by presolve, is finally sent by AMPL to the solver of your choice. The solver is named by AMPL's `solver` option:

In [13]:
ampl.option["solver"]

'highs'

If you have more than one solver, you can switch among them by changing the `solver` option, or by naming the solver in the call to `solve()`, as we have been doing. As an example, we solve the multiperiod production model `steelT` of [Section 4.2](../04/4_2_a_multiperiod_production_model.ipynb) with HiGHS and with CBC, the two solvers installed at the start of this notebook:

In [14]:
%%writefile steelT.mod

set PROD;     # products
param T > 0;  # number of weeks

param rate {PROD} > 0;          # tons per hour produced
param inv0 {PROD} >= 0;         # initial inventory
param avail {1..T} >= 0;        # hours available in week
param market {PROD,1..T} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;     # cost per ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

# Total revenue less costs in all weeks
maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

# Total of hours used by all products
# may not exceed hours available, in each week
subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

# Initial inventory must equal given value
subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

# Tons produced and taken from inventory
# must equal tons sold and put into inventory
subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelT.mod


In [15]:
import pandas as pd

steelT_prod = pd.DataFrame(
    [
        ["bands", 200, 10, 10, 2.5],
        ["coils", 140,  0, 11, 3],
    ],
    columns=["PROD", "rate", "inv0", "prodcost", "invcost"],
).set_index("PROD")

steelT_week = pd.DataFrame(
    [
        ["bands", 1, 25, 6000],
        ["bands", 2, 26, 6000],
        ["bands", 3, 27, 4000],
        ["bands", 4, 27, 6500],
        ["coils", 1, 30, 4000],
        ["coils", 2, 35, 2500],
        ["coils", 3, 37, 3500],
        ["coils", 4, 39, 4200],
    ],
    columns=["PROD", "T", "revenue", "market"],
).set_index(["PROD", "T"])

steelT_avail = {1: 40, 2: 40, 3: 32, 4: 40}

In [16]:
def load_steelT():
    ampl.reset()
    ampl.read("steelT.mod")
    ampl.param["T"] = 4
    ampl.param["avail"] = steelT_avail
    ampl.set_data(steelT_prod, "PROD")
    ampl.set_data(steelT_week)


for solver in ["highs", "cbc"]:
    load_steelT()
    ampl.option["solver"] = solver
    ampl.solve()
    assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations
cbc 2.10.12: optimal solution; objective 515033
0 simplex iterations


In this example we reset the problem between solves, so that the solvers are invoked with the same initial conditions and their performance can be compared. Without `reset()`, information about the solution found by one solver would be passed along to the next one, possibly giving the latter a substantial advantage. Passing information from one solve to the next is most useful when a series of similar LPs are to be sent to the same solver; we discuss this case in more detail in [Section 14.2](../14/14_2_retrieving_results_from_solvers.ipynb). (The function `load_steelT()` and the `for` loop used here are described in [Section 10.5](10_5_running_scripts.ipynb) and [Section 10.6](10_6_iterating_over_a_set_the_for_statement.ipynb).)

Almost any solver can handle a linear program, although those specifically designed for linear programming generally give the best performance. Other kinds of optimization problems, such as nonlinear ([Chapter 18](../18/18.md)) and integer ([Chapter 20](../20/20.md)), can be handled only by solvers designed for them. A message such as "ignoring integrality" or "can't handle nonlinearities" is an indication that you have not chosen a solver appropriate for your model.

If your optimization problems are not too difficult, you should be able to use AMPL without referring to instructions for a specific solver: set the `solver` option appropriately, call `solve()`, and check the result.

If your solver takes a very long time to return with a solution, or returns without any "optimal solution" message, then it's time to read further. Each solver is a sophisticated collection of algorithms and algorithmic strategies, from which many combinations of choices can be made. For most problems the solver makes good choices automatically, but you can also pass along your own choices through an AMPL option named after the solver, such as `highs_options`, or equivalently through a keyword argument of `solve()`. The details vary with each solver, so for more information you must look to the solver-specific documentation, such as the [HiGHS options page](https://dev.ampl.com/solvers/highs/options.html).

If your problem takes a long time to optimize, you will want some evidence of the solver's progress to appear on your screen. Directives for this purpose are also described in the solver-specific documentation; for HiGHS, `outlev=1` displays the solver's log:

In [17]:
load_steelT()
ampl.solve(solver="highs", highs_options="outlev=1")
assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1:   tech:outlev = 1

AMPL MP initial flat model has 24 variables (0 integer, 0 binary);
Objectives: 1 linear; 
Constraints:  12 linear;

AMPL MP did not modify the model.


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP has 12 rows; 24 cols; 38 nonzeros
Coefficient ranges:
  Matrix  [5e-03, 1e+00]
  Cost    [2e+00, 4e+01]
  Bound   [2e+03, 6e+03]
  RHS     [1e+01, 4e+01]
Presolving model
12 rows, 24 cols, 38 nonzeros 0s
Dependent equations search running on 7 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
11 rows, 23 cols, 36 nonzeros 0s
Presolve reductions: rows 11(-1); columns 23(-1); nonzeros 36(-2) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
         16    -5